# Basic Sorting with Insertion Sort


```{contents}
:local:
:depth: 2
```


A payroll report may need employees in name order or invoices in amount order. A sorting algorithm must preserve every occurrence while arranging elements according to a comparison rule. Calling a library sort is useful, but implementing one small sort lets you see the work behind that operation.

Insertion sort is the core teaching algorithm in this chapter. It keeps a sorted prefix and inserts one new value into that prefix. Advanced sorting implementations and detailed comparisons remain in Part V.


## Insert into a Sorted Prefix

```{index} Insert into a Sorted Prefix
```


At the start of iteration i, positions zero through i-1 are sorted and contain the same occurrences as before. Save values[i] before shifting anything. Move larger prefix values one position to the right, then store the saved value in the gap.

For [30,10,20], insert 10 into [30] to obtain [10,30,20]. Then save 20, shift 30, and place 20 between 10 and 30. The array is never extended or shortened. The saved current value prevents shifts from overwriting the item being inserted.

| Pass | Sorted prefix afterward | Remaining values |
| --- | --- | --- |
| Start | 30 | 10, 20 |
| Insert 10 | 10, 30 | 20 |
| Insert 20 | 10, 20, 30 | None |

Use a strict greater-than comparison when shifting. Equal keys stay in their original relative order, which makes this implementation stable.


In [1]:
using System;
using System.Collections.Generic;

int[] amounts = {30, 10, 20, 10};
long comparisons = BasicSort.InsertionSort(amounts, Comparer<int>.Default);
Console.WriteLine(string.Join(", ", amounts));
Console.WriteLine($"Comparisons: {comparisons}");

public static class BasicSort
{
    public static long InsertionSort<T>(T[] values, IComparer<T> comparer)
    {
        long comparisons = 0;
        for (int i = 1; i < values.Length; i++)
        {
            T current = values[i];
            int position = i;
            while (position > 0)
            {
                comparisons++;
                if (comparer.Compare(values[position - 1], current) <= 0) break;
                values[position] = values[position - 1];
                position--;
            }
            values[position] = current;
        }
        return comparisons;
    }
}


10, 10, 20, 30
Comparisons: 6


## Sorting Objects Uses an Ordering Policy

```{index} Sorting Objects Uses an Ordering Policy
```


The comparer returns a negative value, zero, or a positive value. These mean before, tied, or after; callers should test the sign rather than assume the magnitude is exactly one. Define ordering separately from object identity and business equality.

The invoice comparer orders by amount only. Two invoices with the same amount are tied even though they have different identifiers. Stable sorting retains their arrival order. If the business requires amount then ID, include that second key in the comparer rather than relying on an accidental order.


In [1]:
using System;
using System.Collections.Generic;

Invoice[] invoices = {new Invoice("A", 20m), new Invoice("B", 10m), new Invoice("C", 20m)};
BasicSort.InsertionSort(invoices, new AmountComparer());
foreach (Invoice invoice in invoices) Console.WriteLine($"{invoice.Id}: {invoice.Amount:F2}");

public static class BasicSort
{
    public static long InsertionSort<T>(T[] values, IComparer<T> comparer)
    {
        long comparisons = 0;
        for (int i = 1; i < values.Length; i++)
        {
            T current = values[i];
            int position = i;
            while (position > 0)
            {
                comparisons++;
                if (comparer.Compare(values[position - 1], current) <= 0) break;
                values[position] = values[position - 1];
                position--;
            }
            values[position] = current;
        }
        return comparisons;
    }
}

public class Invoice
{
    public string Id { get; }
    public decimal Amount { get; }
    public Invoice(string id, decimal amount) { Id=id; Amount=amount; }
}
public class AmountComparer : IComparer<Invoice>
{
    public int Compare(Invoice? left, Invoice? right)
    {
        if (left is null || right is null) throw new ArgumentException("Invoices required.");
        return left.Amount.CompareTo(right.Amount);
    }
}


B: 10.00
A: 20.00
C: 20.00


## Count Work for Different Inputs

```{index} Count Work for Different Inputs
```


On already sorted data, each new item needs one failed shift comparison, for n-1 comparisons when n is positive. On distinct reverse-sorted data, item i crosses the entire prefix: 1+2+...+(n-1) = n(n-1)/2 comparisons. Therefore the implementation has linear best-case and quadratic worst-case comparison work.

The array is sorted in place with constant auxiliary element storage. Shifting still performs writes, so comparisons are not the only possible cost model. Use the counter to test a specific comparison claim; do not confuse the counter with wall-clock time.


In [1]:
using System;
using System.Collections.Generic;

foreach (int[] input in new[] {Array.Empty<int>(), new[]{1}, new[]{1,2,3,4}, new[]{4,3,2,1}})
{
    long count = BasicSort.InsertionSort(input, Comparer<int>.Default);
    Console.WriteLine($"n={input.Length}, comparisons={count}, result={string.Join(",",input)}");
}

public static class BasicSort
{
    public static long InsertionSort<T>(T[] values, IComparer<T> comparer)
    {
        long comparisons = 0;
        for (int i = 1; i < values.Length; i++)
        {
            T current = values[i];
            int position = i;
            while (position > 0)
            {
                comparisons++;
                if (comparer.Compare(values[position - 1], current) <= 0) break;
                values[position] = values[position - 1];
                position--;
            }
            values[position] = current;
        }
        return comparisons;
    }
}


n=0, comparisons=0, result=
n=1, comparisons=0, result=1
n=4, comparisons=3, result=1,2,3,4
n=4, comparisons=6, result=1,2,3,4


## Verify Order and Preservation

```{index} Verify Order and Preservation
```


A sorted result alone is insufficient: a broken implementation could discard a value and still look ordered. Check length and occurrence preservation as well as ordering. Use a separately sorted copy as a small reference, and include duplicates, negative values, empty input, a singleton, and already sorted data.

In application code, normally use the library operation unless a requirement justifies a custom algorithm. Array.Sort mutates its array and does not promise stable ties. Copy before sorting when callers need the original sequence. Part V investigates additional sorting algorithms and their tradeoffs.


## Practice: Preserve the Saved Item

```{index} Practice: Preserve the Saved Item
```


Complete the final insertion after shifting larger values. Do not lose the saved current item.


In [ ]:
using System;
using System.Collections.Generic;

int[] amounts = {30, 10, 20};
BasicSort.InsertionSort(amounts, Comparer<int>.Default);
Console.WriteLine(string.Join(",", amounts));

public static class BasicSort
{
    public static long InsertionSort<T>(T[] values, IComparer<T> comparer)
    {
        long comparisons = 0;
        for (int i = 1; i < values.Length; i++)
        {
            T current = values[i];
            int position = i;
            while (position > 0)
            {
                comparisons++;
                if (comparer.Compare(values[position - 1], current) <= 0) break;
                values[position] = values[position - 1];
                position--;
            }
            // Your code starts here.
            /* TODO: store current in the gap. */
            // Your code ends here.
        }
        return comparisons;
    }
}


In [1]:
using System;
using System.Collections.Generic;

int[] amounts = {30, 10, 20};
BasicSort.InsertionSort(amounts, Comparer<int>.Default);
Console.WriteLine(string.Join(",", amounts));

public static class BasicSort
{
    public static long InsertionSort<T>(T[] values, IComparer<T> comparer)
    {
        long comparisons = 0;
        for (int i = 1; i < values.Length; i++)
        {
            T current = values[i];
            int position = i;
            while (position > 0)
            {
                comparisons++;
                if (comparer.Compare(values[position - 1], current) <= 0) break;
                values[position] = values[position - 1];
                position--;
            }
            values[position] = current;
        }
        return comparisons;
    }
}


10,20,30


```{rubric} Footnotes
```
[^1]: Stability concerns the relative order of equal comparison keys. It does not mean every object is deeply copied or that objects with the same key have the same identity.
